## MLP

In [3]:
import torch
import sys
import torch.nn as nn
import torch.optim as optim
sys.path.append("../")

from datasets.dataloader import get_wireless_dataloader

# Define Positional Encoding (Fixed for Batch Processing)
class PositionalEncoding(nn.Module):
    def __init__(self, num_freqs=10, input_dim=3):
        super(PositionalEncoding, self).__init__()
        self.num_freqs = num_freqs
        # Each input dimension is encoded as:
        # original value + sin(2^i * x) + cos(2^i * x) for i in 0,...,num_freqs-1
        # Hence, output_dim = input_dim * (1 + 2*num_freqs)
        self.output_dim = input_dim * (2 * num_freqs + 1)

    def forward(self, x):
        """Handles batched inputs (B, 3) correctly"""
        B, D = x.shape  # B: Batch size, D: Input dim (should be 3)
        freqs = 2 ** torch.arange(self.num_freqs, dtype=torch.float32, device=x.device)  # [num_freqs]
        # Compute scaled versions for sin and cos terms: shape becomes [B, 3, num_freqs]
        x_expanded = x.unsqueeze(-1) * freqs  
        # Include the original x only once per dimension: shape [B, 3, 1]
        x_original = x.unsqueeze(-1)
        # Concatenate the original, sine, and cosine terms along the last dimension.
        x_encoded = torch.cat([x_original, torch.sin(x_expanded), torch.cos(x_expanded)], dim=-1)
        return x_encoded.view(B, -1)  # Flatten the last two dimensions

# Define MLP Model
class MLPChannelEstimator(nn.Module):
    def __init__(self, input_dim=3, num_freqs=10, hidden_dim=128, tx_ant=16, rx_ant=2):
        super(MLPChannelEstimator, self).__init__()
        self.tx_ant = tx_ant
        self.rx_ant = rx_ant
        self.pos_encoding = PositionalEncoding(num_freqs=num_freqs, input_dim=input_dim)
        encoded_dim = self.pos_encoding.output_dim

        # MLP with skip connections
        self.fc1 = nn.Linear(encoded_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.fc3 = nn.Linear(hidden_dim, hidden_dim)
        self.fc4 = nn.Linear(hidden_dim + encoded_dim, hidden_dim)  # Skip connection
        self.fc5 = nn.Linear(hidden_dim, hidden_dim)
        self.fc6 = nn.Linear(hidden_dim, hidden_dim)
        self.fc7 = nn.Linear(hidden_dim, hidden_dim)
        self.fc8 = nn.Linear(hidden_dim, tx_ant * rx_ant * 2)  # Output for I, Q pairs

    def forward(self, x):
        """Now supports batch input (B, 3)"""
        x = self.pos_encoding(x)  # Apply positional encoding -> (B, encoded_dim)
        x1 = torch.relu(self.fc1(x))
        x2 = torch.relu(self.fc2(x1))
        x3 = torch.relu(self.fc3(x2))
        # Concatenate the original encoded input with the features from fc3 (skip connection)
        x4 = torch.cat([x, x3], dim=-1)
        x4 = torch.relu(self.fc4(x4))
        x5 = torch.relu(self.fc5(x4))
        x6 = torch.relu(self.fc6(x5))
        x7 = torch.relu(self.fc7(x6))
        x_out = self.fc8(x7)
        return x_out.view(x.shape[0], self.tx_ant, self.rx_ant, 2)  # Reshape to (B, tx_ant, rx_ant, 2)

# Load Dataset
dataloader = get_wireless_dataloader(
    "../datasets/outputs/conf_16x2_414u_5.0ghz_sbrRT_sc104.mat",
    batch_size=16,
    num_pc=16378,
    drop_last=True
)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model, Loss, Optimizer
# Note: Based on your dataset, there are 16 tx antennas and 2 rx antennas.
tx_ant, rx_ant = 16, 2
model = MLPChannelEstimator(tx_ant=tx_ant, rx_ant=rx_ant).to(device)
criterion = nn.MSELoss()  # Mean Squared Error for (I, Q) pairs
optimizer = optim.Adam(model.parameters(), lr=5e-4)

# Training Loop
num_epochs = 50  # Adjust as needed
for epoch in range(num_epochs):
    total_loss = 0.0
    for batch in dataloader:
        rx_positions = batch["rx_position"].to(device)  # [B, 3]
        channel_matrix = batch["channel_matrix"].to(device)  # [B, tx_ant, rx_ant, 2]
        
        # Forward pass (Pass full batch to model)
        predicted_channel = model(rx_positions)  # [B, 16, 2, 2]

        # Compute loss
        loss = criterion(predicted_channel, channel_matrix)

        # Backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.6f}")

# Save Model
torch.save(model.state_dict(), "mlp_channel_estimator.pth")
print("Training Complete. Model saved.")


Epoch [1/50], Loss: 0.001215
Epoch [2/50], Loss: 0.000066
Epoch [3/50], Loss: 0.000017
Epoch [4/50], Loss: 0.000010
Epoch [5/50], Loss: 0.000008
Epoch [6/50], Loss: 0.000007
Epoch [7/50], Loss: 0.000007
Epoch [8/50], Loss: 0.000006
Epoch [9/50], Loss: 0.000006
Epoch [10/50], Loss: 0.000005
Epoch [11/50], Loss: 0.000005
Epoch [12/50], Loss: 0.000005
Epoch [13/50], Loss: 0.000005
Epoch [14/50], Loss: 0.000005
Epoch [15/50], Loss: 0.000005
Epoch [16/50], Loss: 0.000005
Epoch [17/50], Loss: 0.000005
Epoch [18/50], Loss: 0.000004
Epoch [19/50], Loss: 0.000004
Epoch [20/50], Loss: 0.000004
Epoch [21/50], Loss: 0.000004
Epoch [22/50], Loss: 0.000004
Epoch [23/50], Loss: 0.000004
Epoch [24/50], Loss: 0.000004
Epoch [25/50], Loss: 0.000004
Epoch [26/50], Loss: 0.000004
Epoch [27/50], Loss: 0.000004
Epoch [28/50], Loss: 0.000004
Epoch [29/50], Loss: 0.000004
Epoch [30/50], Loss: 0.000003
Epoch [31/50], Loss: 0.000003
Epoch [32/50], Loss: 0.000003
Epoch [33/50], Loss: 0.000003
Epoch [34/50], Loss

## Inference